# OpenPlaque — Left Proximal-Trunk Continuation QC v1
Dense source-CCTA adjudication of the strongly supported vessel path found beyond the accepted LAD endpoint. A positive result validates a proximal-trunk continuation candidate; it does **not** establish LM identity or modify the frozen master.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_DIR=DRIVE_ROOT+'/Left_Proximal_Trunk_Continuation_QC_v1'
BRANCH='left-proximal-trunk-continuation-qc-from-main'
PINNED_SCIENCE_COMMIT='100b4803977850371f0f07ee9a8804620fded0e3'
BASELINE='0593b453959f5a353d644267fbeef24b514ef4d7'
EXPECTED_ALGORITHM='left-proximal-trunk-continuation-qc-v1.0'
print('Branch:',BRANCH)
print('Pinned science commit:',PINNED_SCIENCE_COMMIT)
print('Output:',OUTPUT_DIR)


In [ ]:
import os,shutil,subprocess
repo='/content/OpenPlaque'
if os.path.exists(repo): shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','20','--branch',BRANCH,'https://github.com/pazzani/OpenPlaque.git',repo],check=True)
subprocess.run(['git','-C',repo,'checkout','--detach',PINNED_SCIENCE_COMMIT],check=True)
HEAD=subprocess.check_output(['git','-C',repo,'rev-parse','HEAD'],text=True).strip()
print('Checked out HEAD:',HEAD)
assert HEAD==PINNED_SCIENCE_COMMIT,(HEAD,PINNED_SCIENCE_COMMIT)
mb=subprocess.check_output(['git','-C',repo,'merge-base','HEAD',BASELINE],text=True).strip()
print('Merge base:',mb)
assert mb==BASELINE


In [ ]:
%pip uninstall -y openplaque >/dev/null 2>&1
%pip install -q --no-cache-dir --force-reinstall --no-deps /content/OpenPlaque
%pip install -q pytest SimpleITK scipy pandas matplotlib numpy pydicom psutil


In [ ]:
import sys,importlib,pathlib,pytest
for name in list(sys.modules):
    if name=='openplaque' or name.startswith('openplaque.'):
        del sys.modules[name]
importlib.invalidate_caches()
import openplaque
from openplaque import left_proximal_trunk_continuation_qc_v1 as exp
print('openplaque:',openplaque.__file__)
print('experiment:',exp.__file__)
print('algorithm:',exp.ALGORITHM)
assert exp.BASELINE==BASELINE
assert exp.ALGORITHM==EXPECTED_ALGORITHM
compile(pathlib.Path(exp.__file__).read_text(),exp.__file__,'exec')
print('synthetic truncation:',exp.synthetic_truncation_self_test())
rc=pytest.main(['-q','/content/OpenPlaque/tests/test_left_proximal_trunk_continuation_qc_v1.py'])
if rc!=0: raise RuntimeError(f'pytest failed with exit code {rc}')


In [ ]:
from pathlib import Path
import json
root=Path(DRIVE_ROOT)
required=[root/'Left_Main_LAD_Mask_Gate_Diagnostic_v1/summary.json',root/'Left_Main_LAD_Mask_Gate_Diagnostic_v1/hard_mask_best_frontier_path.csv',root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.npy',root/'Cache/Secondary_3D_Vesselness_Topology_v1/series7_int16.json',root/'Cache/LAD_Frozen_Proximal_Reacquisition_v1/combined_lad_centerline.csv',root/'PCAT_RCA_10_50/rca_centerline_smoothed_zyx.csv',root/'TotalSegmentator_Cardiovascular_Cache_v1/heartchambers_highres/aorta.nii.gz',root/'Cache/Master_Coronary_Anatomy_Baseline_v2/master_anatomy_summary.json']
missing=[str(p) for p in required if not p.exists()]
print('Preflight required:',len(required),'missing:',len(missing))
if missing: raise FileNotFoundError('\n'.join(missing))
prior=json.loads(required[0].read_text())
print('Prior diagnostic status:',prior.get('status'))
print('Hard search:',prior.get('hard_mask_search'))
print('Shadow search:',prior.get('shadow_soft_mask_search'))


In [ ]:
import time,gc
from openplaque.left_proximal_trunk_continuation_qc_v1 import run
gc.collect(); t0=time.time()
result=run(DRIVE_ROOT,OUTPUT_DIR)
s=result['summary']
print('ELAPSED MIN:',round((time.time()-t0)/60,2))
print('STATUS:',s.get('status'))
print('RCA PLANE QC:',s.get('RCA_plane_qc_control'))
print('CONTINUATION QC:',s.get('candidate_dense_qc'))
print('ROLE:',s.get('candidate_role'))
print('REPORT:',result.get('report'))
print('ZIP:',result.get('zip'))
